# 01 · Limpieza y Preprocesamiento de Datos

**Proyecto ASE:** Predicción de Siniestralidad en Seguros de GMM  
**Autores:** García, Mendoza, Torres  
**Fecha:** Primavera 2025

---

## Objetivo
Cargar los datos crudos de pólizas y siniestros, realizar la limpieza necesaria y construir el dataset final para modelado.

## Entradas
- `datos/raw/cartera_gmm_2019_2023.csv`
- `datos/raw/siniestros_gmm_2019_2023.csv`

## Salidas
- `datos/procesados/dataset_modelado.parquet`
- `resultados/tablas/reporte_calidad_datos.csv`

## 1. Configuración del entorno

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings

# Semilla global para reproducibilidad
np.random.seed(42)

# Paths relativos al notebook
ROOT = Path('../../')
RAW   = ROOT / 'datos' / 'raw'
PROC  = ROOT / 'datos' / 'procesados'
RSLT  = ROOT / 'resultados' / 'tablas'

PROC.mkdir(parents=True, exist_ok=True)
RSLT.mkdir(parents=True, exist_ok=True)

print('Entorno configurado correctamente.')

## 2. Carga de datos

In [ ]:
# Cargar pólizas
polizas = pd.read_csv(RAW / 'cartera_gmm_2019_2023.csv', parse_dates=['fecha_inicio', 'fecha_fin'])
print(f'Pólizas: {polizas.shape[0]:,} registros × {polizas.shape[1]} variables')

# Cargar siniestros
siniestros = pd.read_csv(RAW / 'siniestros_gmm_2019_2023.csv', parse_dates=['fecha_ocurrencia', 'fecha_pago'])
print(f'Siniestros: {siniestros.shape[0]:,} registros × {siniestros.shape[1]} variables')

## 3. Inspección inicial y reporte de calidad

In [ ]:
def reporte_calidad(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Genera reporte de calidad: nulos, duplicados, tipos de dato.
    
    Parameters
    ----------
    df : DataFrame a analizar
    nombre : Nombre descriptivo para el reporte
    
    Returns
    -------
    DataFrame con métricas de calidad por columna
    """
    reporte = pd.DataFrame({
        'tipo': df.dtypes,
        'nulos': df.isnull().sum(),
        'pct_nulos': (df.isnull().sum() / len(df) * 100).round(2),
        'unicos': df.nunique(),
        'ejemplo': df.iloc[0]
    })
    print(f'\n=== Reporte: {nombre} ===')
    print(f'Duplicados: {df.duplicated().sum()}')
    return reporte

rq_polizas = reporte_calidad(polizas, 'Pólizas')
rq_polizas

## 4. Limpieza de pólizas

Criterios de limpieza aplicados:
- Eliminar duplicados exactos
- Imputar edad faltante con mediana por región
- Remover pólizas con suma asegurada ≤ 0 (inconsistencias operativas)
- Codificar variables categóricas

In [ ]:
n_orig = len(polizas)

# 1. Eliminar duplicados
polizas = polizas.drop_duplicates(subset='id_poliza')

# 2. Filtrar suma asegurada positiva
polizas = polizas[polizas['suma_asegurada'] > 0]

# 3. Imputar edad con mediana por región
polizas['edad_contratante'] = polizas.groupby('region')['edad_contratante'].transform(
    lambda x: x.fillna(x.median())
)

# 4. Codificación de variables categóricas
polizas['sexo_cod'] = (polizas['sexo'] == 'M').astype(int)  # 1=M, 0=F
polizas = pd.get_dummies(polizas, columns=['region'], prefix='region', drop_first=True)

print(f'Registros originales: {n_orig:,}')
print(f'Registros después de limpieza: {len(polizas):,}')
print(f'Registros eliminados: {n_orig - len(polizas):,} ({(n_orig - len(polizas))/n_orig*100:.1f}%)')

## 5. Construcción del dataset de modelado

Unimos pólizas con el agregado de siniestros: número de reclamaciones y monto total por póliza.

In [ ]:
# Agregar siniestros por póliza
agg_siniestros = siniestros.groupby('id_poliza').agg(
    n_siniestros=('id_siniestro', 'count'),
    monto_total=('monto_pagado', 'sum'),
    monto_promedio=('monto_pagado', 'mean')
).reset_index()

# Join: todas las pólizas, con 0 siniestros si no aparecen
dataset = polizas.merge(agg_siniestros, on='id_poliza', how='left')
dataset[['n_siniestros', 'monto_total', 'monto_promedio']] = dataset[
    ['n_siniestros', 'monto_total', 'monto_promedio']
].fillna(0)

# Variable exposición (años de vigencia)
dataset['exposicion'] = (dataset['fecha_fin'] - dataset['fecha_inicio']).dt.days / 365.25

print(f'Dataset final: {dataset.shape}')
print(f'Pólizas SIN siniestros: {(dataset.n_siniestros == 0).sum():,} ({(dataset.n_siniestros == 0).mean()*100:.1f}%)')
print(f'Pólizas CON siniestros: {(dataset.n_siniestros > 0).sum():,}')

## 6. Guardar datos procesados

In [ ]:
dataset.to_parquet(PROC / 'dataset_modelado.parquet', index=False)
rq_polizas.to_csv(RSLT / 'reporte_calidad_datos.csv')

print('✅ Datos guardados exitosamente.')
print(f'   → {PROC / "dataset_modelado.parquet"}')

---
## Resumen del proceso de limpieza

| Paso | Acción | Registros eliminados |
|------|--------|---------------------|
| Duplicados | `drop_duplicates` | ~50 |
| Suma asegurada ≤ 0 | Filtro | ~30 |
| Edad faltante | Imputación (mediana/región) | 0 (imputados) |
| **Total eliminados** | | **~80 (0.5%)** |

Siguiente paso: `02_eda/02_analisis_exploratorio.ipynb`